In [15]:
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id="ibm-esa-geospatial/TerraMind-1.0-small")
print("Downloaded to:", path)

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Downloaded to: /Users/mayankthakre/.cache/huggingface/hub/models--ibm-esa-geospatial--TerraMind-1.0-small/snapshots/960f7549bfe9d7a08946860042f83badd604c779


In [17]:
import torch
from terratorch import BACKBONE_REGISTRY

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Rebuild with ONLY the 2 bands we actually have
# SWIR_1 = B11, SWIR_2 = B12
terramind = BACKBONE_REGISTRY.build(
    'terramind_v1_small',
    pretrained=True,
    modalities=['S2L2A'],
    bands={'S2L2A': ['SWIR_1', 'SWIR_2']}   # 2-band subset
)

terramind = terramind.to(device).float()
terramind.eval()

# Freeze ALL weights — we only train our heads
for param in terramind.parameters():
    param.requires_grad = False

frozen = sum(p.numel() for p in terramind.parameters() if not p.requires_grad)
print(f"TerraMind-small loaded with SWIR_1 + SWIR_2 only")
print(f"Frozen parameters: {frozen:,}")

2026-04-26 00:14:21,423 - INFO - Loaded weights for SWIR_1 in position 0 of patch embed
2026-04-26 00:14:21,425 - INFO - Loaded weights for SWIR_2 in position 1 of patch embed


TerraMind-small loaded with SWIR_1 + SWIR_2 only
Frozen parameters: 21,481,728


In [18]:
# Step 1: Understand what input format the model expects
print("=== Model input inspection ===")
print("Type:", type(terramind).__name__)

# Check encoder embeddings to understand expected input key
if hasattr(terramind, 'encoder_embeddings'):
    print("Encoder modality keys:", list(terramind.encoder_embeddings.keys()))

# Check patch embed to understand spatial expectations
if hasattr(terramind, 'patch_embed'):
    print("Patch embed:", terramind.patch_embed)

=== Model input inspection ===
Type: TerraMindViT
Encoder modality keys: ['untok_sen2l2a@224']


In [21]:
# TerraMind ViT: patch_size=16, img=224 → 14×14=196 patches
# Input dict key must match exactly what encoder_embeddings has

# Try direct tensor first (some ViT variants accept plain tensor)
dummy_tensor = torch.randn(1, 2, 224, 224).to(device).float()
dummy_dict   = {'S2L2A': dummy_tensor}

for fmt_name, inp in [("plain tensor", dummy_tensor), ("dict S2L2A", dummy_dict)]:
    try:
        with torch.no_grad():
            out = terramind(inp)
        if isinstance(out, (list, tuple)):
            shapes = [o.shape for o in out if hasattr(o, 'shape')]
            print(f" {fmt_name} → {shapes}")
        elif isinstance(out, dict):
            print(f" {fmt_name} → {{{', '.join(f'{k}:{v.shape}' for k,v in out.items() if hasattr(v,'shape'))}}}")
        elif hasattr(out, 'shape'):
            print(f" {fmt_name} → {out.shape}")
        break
    except Exception as e:
        print(f"{fmt_name}: {str(e)[:100]}")

 plain tensor → [torch.Size([1, 196, 384]), torch.Size([1, 196, 384]), torch.Size([1, 196, 384]), torch.Size([1, 196, 384]), torch.Size([1, 196, 384]), torch.Size([1, 196, 384]), torch.Size([1, 196, 384]), torch.Size([1, 196, 384]), torch.Size([1, 196, 384]), torch.Size([1, 196, 384]), torch.Size([1, 196, 384]), torch.Size([1, 196, 384])]


In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TerraMindMethanePulse(nn.Module):
    def __init__(self, encoder, embed_dim=384, img_size=512):
        super().__init__()
        self.encoder  = encoder     # frozen TerraMind
        self.img_size = img_size

        # Segmentation decoder: 14×14 → 512×512
        self.seg_decoder = nn.Sequential(
            nn.ConvTranspose2d(embed_dim, 256, kernel_size=2, stride=2),  # 28×28
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2),        # 56×56
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64,  kernel_size=2, stride=2),        # 112×112
            nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64,  32,  kernel_size=2, stride=2),        # 224×224
            nn.BatchNorm2d(32),  nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, kernel_size=1)                               # logits
        )
        # Final upsample 224→512 done in forward via interpolate

        # Severity head: pooled embedding → 3 classes
        self.severity_head = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 3)   # 0=low, 1=medium, 2=high
        )

    def forward(self, x):
        # x: (B, 2, 512, 512) — B11, B12
        B = x.shape[0]

        # Resize to 224×224 for TerraMind
        x_224 = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)

        # TerraMind forward — returns list of 12 layer outputs
        with torch.no_grad():
            features = self.encoder(x_224)   # list of [B, 196, 384]

        # Use last layer features
        feat = features[-1]                  # (B, 196, 384)

        # Reshape to spatial feature map: 196 = 14×14
        # AFTER (fixed)
        feat_map = feat.permute(0, 2, 1).contiguous().reshape(B, 384, 14, 14)

        # Segmentation branch
        seg_logits = self.seg_decoder(feat_map)                    
        seg_logits = F.interpolate(seg_logits, size=(512, 512),
                                   mode='bilinear', align_corners=False)  # (B, 1, 512, 512)

        # Severity branch
        pooled        = feat.mean(dim=1)                           # (B, 384)
        severity_logits = self.severity_head(pooled)               # (B, 3)

        return seg_logits, severity_logits


# Instantiate
model = TerraMindMethanePulse(encoder=terramind, embed_dim=384, img_size=512)
model = model.to(device).float()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f" TerraMindMethanePulse ready")
print(f"Total params     : {total:,}")
print(f"Trainable params : {trainable:,}  (heads only — encoder frozen)")
print(f"Frozen params    : {total - trainable:,}  (TerraMind)")

 TerraMindMethanePulse ready
Total params     : 22,098,116
Trainable params : 616,388  (heads only — encoder frozen)
Frozen params    : 21,481,728  (TerraMind)


In [25]:
# Reinitialize only the severity head
def reinit_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)

model.severity_head.apply(reinit_weights)
print("Severity head reinitialized")
print(f"Seg decoder first weight mean: {model.seg_decoder[0].weight.mean().item():.6f}")

Severity head reinitialized
Seg decoder first weight mean: -0.000014


In [27]:
dummy = torch.randn(2, 2, 512, 512).to(device).float()
seg_out, sev_out = model(dummy)
print(f"Seg output  : {seg_out.shape}   ← expected (2, 1, 512, 512)")
print(f"Sev output  : {sev_out.shape}    ← expected (2, 3)")
print("Forward pass confirmed")

Seg output  : torch.Size([2, 1, 512, 512])   ← expected (2, 1, 512, 512)
Sev output  : torch.Size([2, 3])    ← expected (2, 3)
Forward pass confirmed


In [29]:
import numpy as np
import pandas as pd
import rasterio
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from collections import Counter

RAW       = Path("data/raw")
label_dir = RAW / "label"

def pair_image_label(img_path, label_dir):
    stem = img_path.stem.replace("_nonorm_bands20m", "")
    return Path(label_dir) / f"{stem}_bin_label.tiff"

# ── Build train/val splits first (needed for thresholds) ─────────
all_files = sorted(RAW.glob("*.tiff"))
records = []
for f in all_files:
    main = f.stem.split("___")
    records.append({"path": f, "sample_id": int(main[0]),
                    "location_id": int(main[1].split("_")[0])})

meta_df  = pd.DataFrame(records)
train_df = meta_df[~meta_df["location_id"].isin([8, 9])].reset_index(drop=True)
val_df   = meta_df[meta_df["location_id"].isin([8, 9])].reset_index(drop=True)

# ── Compute data-driven thresholds from train set ─────────────────
all_pixel_counts = []
for _, r in train_df.iterrows():
    lbl_path = pair_image_label(r["path"], label_dir)
    with rasterio.open(lbl_path) as src:
        mask_np = (src.read(1) > 0.5).astype(float)
    all_pixel_counts.append(int(mask_np.sum()))

all_pixel_counts = np.array(all_pixel_counts)
P33 = int(np.percentile(all_pixel_counts, 33))
P66 = int(np.percentile(all_pixel_counts, 66))
print(f"Thresholds → Low: <{P33}px | Medium: {P33}–{P66}px | High: >{P66}px")

def derive_severity(mask_np):
    plume_px = mask_np.sum()
    if   plume_px < P33: return 0
    elif plume_px < P66: return 1
    else:                return 2

class TerraMindMethaneDataset(Dataset):
    def __init__(self, dataframe, label_dir):
        self.df        = dataframe.reset_index(drop=True)
        self.label_dir = Path(label_dir)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = row["path"]
        lbl_path = pair_image_label(img_path, self.label_dir)
        eps      = 1e-6

        with rasterio.open(img_path) as src:
            img = src.read().astype(np.float32)

        # Only B11 and B12 — exactly what TerraMind SWIR_1/SWIR_2 expects
        b11, b12 = img[11], img[12]

        def norm(x):
            lo, hi = np.percentile(x, 2), np.percentile(x, 98)
            return np.clip((x - lo) / (hi - lo + eps), 0, 1).astype(np.float32)

        x = np.stack([norm(b11), norm(b12)], axis=0)   # (2, H, W)

        with rasterio.open(lbl_path) as src:
            lbl  = src.read(1).astype(np.float32)
        mask = (lbl > 0.5).astype(np.float32)

        severity = derive_severity(mask)

        return (torch.tensor(x,    dtype=torch.float32),
                torch.tensor(mask, dtype=torch.float32).unsqueeze(0),
                torch.tensor(severity, dtype=torch.long))


# Build train/val splits
all_files = sorted(RAW.glob("*.tiff"))
records = []
for f in all_files:
    main = f.stem.split("___")
    records.append({"path": f, "sample_id": int(main[0]),
                    "location_id": int(main[1].split("_")[0])})

meta_df   = pd.DataFrame(records)
train_df  = meta_df[~meta_df["location_id"].isin([8, 9])].reset_index(drop=True)
val_df    = meta_df[meta_df["location_id"].isin([8, 9])].reset_index(drop=True)

train_dataset = TerraMindMethaneDataset(train_df, label_dir)
val_dataset   = TerraMindMethaneDataset(val_df,   label_dir)

train_loader  = DataLoader(train_dataset, batch_size=2, shuffle=True,  num_workers=0)
val_loader    = DataLoader(val_dataset,   batch_size=2, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

sev_labels = []
for _, r in val_df.iterrows():
    lbl_path = pair_image_label(r["path"], label_dir)
    with rasterio.open(lbl_path) as src:
        mask_np = (src.read(1) > 0.5).astype(float)
    sev_labels.append(derive_severity(mask_np))

print("Val severity distribution:", Counter(sev_labels))
print("  0=low, 1=medium, 2=high")
print("Dataset ready")


Thresholds → Low: <932px | Medium: 932–940px | High: >940px
Train: 619 | Val: 185
Val severity distribution: Counter({2: 73, 0: 62, 1: 50})
  0=low, 1=medium, 2=high
Dataset ready


In [30]:
def dice_loss(pred, target, eps=1e-6):
    pred   = torch.sigmoid(pred)
    inter  = (pred * target).sum(dim=(2, 3))
    union  = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    return 1 - ((2 * inter + eps) / (union + eps)).mean()

def combined_seg_loss(logits, target):
    bce  = nn.BCEWithLogitsLoss()(logits, target)
    dice = dice_loss(logits, target)
    return 0.5 * bce + 0.5 * dice

sev_criterion = nn.CrossEntropyLoss()

print("Loss functions defined")

Loss functions defined


In [10]:
# Phase 1: severity-only warmup for 10 epochs
print("Phase 1: Severity head warmup")

opt_sev = torch.optim.Adam(model.severity_head.parameters(), lr=1e-3)

for ep in range(1, 11):
    model.train()
    correct = total = 0
    for x, mask, severity in train_loader:
        x        = x.to(device, dtype=torch.float32)
        severity = severity.to(device)

        opt_sev.zero_grad()
        _, sev_logits = model(x)
        loss = sev_criterion(sev_logits, severity)
        loss.backward()
        opt_sev.step()

        correct += (sev_logits.argmax(1) == severity).sum().item()
        total   += severity.size(0)

    print(f"  Warmup epoch {ep:>2}/10 | train sev_acc: {correct/total:.3f}")

print("Phase 1 complete — now run Cell 9 for joint training")

Phase 1: Severity head warmup
  Warmup epoch  1/10 | train sev_acc: 0.341
  Warmup epoch  2/10 | train sev_acc: 0.373
  Warmup epoch  3/10 | train sev_acc: 0.373
  Warmup epoch  4/10 | train sev_acc: 0.338
  Warmup epoch  5/10 | train sev_acc: 0.342
  Warmup epoch  6/10 | train sev_acc: 0.367
  Warmup epoch  7/10 | train sev_acc: 0.376
  Warmup epoch  8/10 | train sev_acc: 0.376
  Warmup epoch  9/10 | train sev_acc: 0.381
  Warmup epoch 10/10 | train sev_acc: 0.388
Phase 1 complete — now run Cell 9 for joint training


In [14]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
from collections import Counter
import json

# ── Class weights from train set ─────────────────────────────────
train_sev_labels = []
for _, r in train_df.iterrows():
    lbl_path = pair_image_label(r["path"], label_dir)
    with rasterio.open(lbl_path) as src:
        mask_np = (src.read(1) > 0.5).astype(float)
    train_sev_labels.append(derive_severity(mask_np))

train_counts = Counter(train_sev_labels)
total_train  = len(train_sev_labels)
class_weights = torch.tensor([
    total_train / (3 * train_counts.get(i, 1)) for i in range(3)
], dtype=torch.float32).to(device)

print("Train severity distribution:", dict(train_counts))
print("Class weights:", class_weights)

sev_criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,           # was 5e-4
    weight_decay=1e-4
)
scheduler = ReduceLROnPlateau(optimizer, mode='max',
                               patience=5,   # was 3
                               factor=0.5)

EPOCHS = 20   # was 15
best_iou = 0.0
history    = [] 


scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

SEG_WEIGHT = 0.7   # restore seg dominance
SEV_WEIGHT = 0.3
lr         = 5e-4  # lower — head is already warmed up
patience   = 5     # make sure this is 5 not 3
EPOCHS     = 20
history    = [] 

print(f"Training TerraMind heads for {EPOCHS} epochs")
print(f"Loss = {SEG_WEIGHT}×seg + {SEV_WEIGHT}×severity\n")
print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'IoU':>8} | {'Sev Acc':>8} | {'LR':>8}")
print("-" * 65)

for epoch in range(1, EPOCHS + 1):

    # ── Train ──────────────────────────────────────
    model.train()
    train_loss = 0.0
    for x, mask, severity in train_loader:
        x        = x.to(device, dtype=torch.float32)
        mask     = mask.to(device, dtype=torch.float32)
        severity = severity.to(device)

        optimizer.zero_grad()
        seg_logits, sev_logits = model(x)

        loss_seg = combined_seg_loss(seg_logits, mask)
        loss_sev = sev_criterion(sev_logits, severity)
        loss     = SEG_WEIGHT * loss_seg + SEV_WEIGHT * loss_sev

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ── Validate ────────────────────────────────────
    model.eval()
    val_loss = total_iou = sev_correct = sev_total = 0.0

    with torch.no_grad():
        for x, mask, severity in val_loader:
            x        = x.to(device, dtype=torch.float32)
            mask     = mask.to(device, dtype=torch.float32)
            severity = severity.to(device)

            seg_logits, sev_logits = model(x)

            loss_seg = combined_seg_loss(seg_logits, mask)
            loss_sev = sev_criterion(sev_logits, severity)
            val_loss += (SEG_WEIGHT * loss_seg + SEV_WEIGHT * loss_sev).item()

            pred_mask = (torch.sigmoid(seg_logits) > 0.5).float()
            eps = 1e-6
            tp = ((pred_mask==1)&(mask==1)).sum().item()
            fp = ((pred_mask==1)&(mask==0)).sum().item()
            fn = ((pred_mask==0)&(mask==1)).sum().item()
            total_iou += tp / (tp + fp + fn + eps)

            pred_sev     = sev_logits.argmax(dim=1)
            sev_correct += (pred_sev == severity).sum().item()
            sev_total   += severity.size(0)

    val_loss /= len(val_loader)
    val_iou   = total_iou / len(val_loader)
    sev_acc   = sev_correct / sev_total
    cur_lr    = optimizer.param_groups[0]['lr']

    scheduler.step(val_iou)

    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(),
                   "outputs/checkpoints/terramind_methane_best.pt")
        ckpt_mark = " ← best"
    else:
        ckpt_mark = ""

    print(f"{epoch:>6}/{EPOCHS} | {train_loss:>10.4f} | {val_loss:>10.4f} | "
          f"{val_iou:>8.4f} | {sev_acc:>8.3f} | {cur_lr:>8.1e}{ckpt_mark}")

    history.append({"epoch": epoch, "train_loss": train_loss,
                    "val_loss": val_loss, "iou": val_iou, "sev_acc": sev_acc})

with open("outputs/terramind_training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print(f"\n Training complete | Best IoU: {best_iou:.4f}")

Train severity distribution: {2: 231, 0: 174, 1: 214}
Class weights: tensor([1.1858, 0.9642, 0.8932], device='mps:0')
Training TerraMind heads for 20 epochs
Loss = 0.7×seg + 0.3×severity

 Epoch | Train Loss |   Val Loss |      IoU |  Sev Acc |       LR
-----------------------------------------------------------------
     1/20 |     0.5142 |     0.5579 |   0.2920 |    0.395 |  1.0e-03 ← best
     2/20 |     0.4806 |     0.5507 |   0.3107 |    0.395 |  1.0e-03 ← best
     3/20 |     0.4703 |     0.5536 |   0.3101 |    0.346 |  1.0e-03
     4/20 |     0.4637 |     0.5587 |   0.2978 |    0.319 |  1.0e-03
     5/20 |     0.4620 |     0.5435 |   0.3235 |    0.395 |  1.0e-03 ← best
     6/20 |     0.4558 |     0.5493 |   0.3123 |    0.395 |  1.0e-03
     7/20 |     0.4542 |     0.5479 |   0.3187 |    0.346 |  1.0e-03
     8/20 |     0.4502 |     0.5416 |   0.3298 |    0.395 |  1.0e-03 ← best
     9/20 |     0.4513 |     0.5447 |   0.3250 |    0.395 |  1.0e-03
    10/20 |     0.4450 |     0.

In [33]:
import numpy as np

pixel_counts = []
for _, r in train_df.iterrows():
    lbl_path = pair_image_label(r["path"], label_dir)
    with rasterio.open(lbl_path) as src:
        mask_np = (src.read(1) > 0.5).astype(float)
    pixel_counts.append(int(mask_np.sum()))

pixel_counts = np.array(pixel_counts)
print(f"Min   : {pixel_counts.min()}")
print(f"P25   : {np.percentile(pixel_counts, 25):.0f}")
print(f"Median: {np.percentile(pixel_counts, 50):.0f}")
print(f"P75   : {np.percentile(pixel_counts, 75):.0f}")
print(f"Max   : {pixel_counts.max()}")
print(f"\nZero-plume tiles: {(pixel_counts == 0).sum()}")


Min   : 916
P25   : 931
Median: 936
P75   : 1856
Max   : 2830

Zero-plume tiles: 0


In [35]:
%%writefile alert_packet.py
import json
import uuid
import torch
import numpy as np
import rasterio
from datetime import datetime, timezone
from pathlib import Path

SEVERITY_LABELS = {0: "LOW", 1: "MEDIUM", 2: "HIGH"}
SEVERITY_COLOR  = {0: "#f5c518", 1: "#ff8c00", 2: "#e03131"}

def generate_alert_packet(tiff_path, model, device, derive_severity_fn,
                           pair_image_label_fn, label_dir,
                           confidence_threshold=0.5):
    tiff_path = Path(tiff_path)
    with rasterio.open(tiff_path) as src:
        img       = src.read().astype(np.float32)
        crs       = str(src.crs)
        bounds    = src.bounds
        transform = src.transform

    eps = 1e-6
    b11, b12 = img[11], img[12]

    def norm(x):
        lo, hi = np.percentile(x, 2), np.percentile(x, 98)
        return np.clip((x - lo) / (hi - lo + eps), 0, 1).astype(np.float32)

    x_tensor = torch.tensor(
        np.stack([norm(b11), norm(b12)], axis=0),
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        seg_logits, sev_logits = model(x_tensor)

    prob_map  = torch.sigmoid(seg_logits).squeeze().cpu().numpy()
    pred_mask = (prob_map > confidence_threshold).astype(np.uint8)
    sev_class = int(sev_logits.argmax(dim=1).item())
    sev_conf  = float(torch.softmax(sev_logits, dim=1).max().item())

    plume_px  = int(pred_mask.sum())
    total_px  = pred_mask.size
    plume_pct = round(100 * plume_px / total_px, 4)
    mean_prob = round(float(prob_map[pred_mask == 1].mean())
                      if plume_px > 0 else 0.0, 4)

    if plume_px > 0:
        ys, xs       = np.where(pred_mask == 1)
        centroid_px  = [int(xs.mean()), int(ys.mean())]
        cx_geo       = transform.c + centroid_px[0] * transform.a
        cy_geo       = transform.f + centroid_px[1] * transform.e
        centroid_geo = [round(cx_geo, 6), round(cy_geo, 6)]
    else:
        centroid_px  = None
        centroid_geo = None

    gt_iou = None
    gt_severity = None
    try:
        lbl_path = pair_image_label_fn(tiff_path, label_dir)
        with rasterio.open(lbl_path) as src:
            gt_mask = (src.read(1) > 0.5).astype(np.uint8)
        tp = int(((pred_mask == 1) & (gt_mask == 1)).sum())
        fp = int(((pred_mask == 1) & (gt_mask == 0)).sum())
        fn = int(((pred_mask == 0) & (gt_mask == 1)).sum())
        gt_iou      = round(tp / (tp + fp + fn + 1e-6), 4)
        gt_severity = int(derive_severity_fn(gt_mask.astype(float)))
    except Exception:
        pass

    return {
        "alert_id"      : str(uuid.uuid4()),
        "schema_version": "1.0",
        "timestamp_utc" : datetime.now(timezone.utc).isoformat(),
        "source": {
            "tile_path" : str(tiff_path),
            "sensor"    : "Sentinel-2",
            "bands_used": ["B11 (SWIR_1)", "B12 (SWIR_2)"],
            "crs"       : crs,
            "bounds"    : {
                "left"  : round(bounds.left,   6),
                "bottom": round(bounds.bottom, 6),
                "right" : round(bounds.right,  6),
                "top"   : round(bounds.top,    6)
            }
        },
        "detection": {
            "plume_detected"      : bool(plume_px > 0),
            "confidence_threshold": confidence_threshold,
            "plume_pixel_count"   : plume_px,
            "plume_coverage_pct"  : plume_pct,
            "mean_plume_prob"     : mean_prob,
            "centroid_pixel"      : centroid_px,
            "centroid_geographic" : centroid_geo
        },
        "severity": {
            "class_id"  : sev_class,
            "label"     : SEVERITY_LABELS[sev_class],
            "confidence": round(sev_conf, 4),
            "color_hex" : SEVERITY_COLOR[sev_class]
        },
        "model": {
            "name"            : "TerraMindMethanePulse",
            "encoder"         : "TerraMind-1.0-small (frozen)",
            "trainable_params": 616388,
            "best_val_iou"    : 0.3298
        },
        "ground_truth": {
            "iou"          : gt_iou,
            "severity_class": gt_severity
        } if gt_iou is not None else None,
        "status": "ALERT" if plume_px > 0 else "CLEAR"
    }

def save_alert(packet, out_dir="outputs/alerts"):
    out_dir  = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{packet['alert_id']}.json"
    with open(out_path, "w") as f:
        json.dump(packet, f, indent=2)
    print(f"Alert saved → {out_path}")
    return out_path

Overwriting alert_packet.py


In [37]:
import json
import importlib
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

# ── Rebuild model architecture ────────────────────────────────────
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Re-run Cell 5 first OR load weights directly like this:
model = TerraMindMethanePulse(encoder=terramind, embed_dim=384, img_size=512)
model = model.to(device).float()
model.load_state_dict(torch.load(
    "outputs/checkpoints/terramind_methane_best.pt",
    map_location=device
))
model.eval()
print(f"Model loaded on {device}")

# ── Now run the alert test ────────────────────────────────────────
import alert_packet, importlib
importlib.reload(alert_packet)
from alert_packet import generate_alert_packet, save_alert

sample_tile = list(Path("data/raw").glob("*.tiff"))[0]
print(f"Testing on: {sample_tile.name}")

packet = generate_alert_packet(
    tiff_path           = sample_tile,
    model               = model,
    device              = device,
    derive_severity_fn  = derive_severity,
    pair_image_label_fn = pair_image_label,
    label_dir           = label_dir
)

save_alert(packet)
print(json.dumps(packet, indent=2))

Model loaded on mps
Testing on: 10___1_S2A_20240317T085711_R1000_nonorm_bands20m.tiff
Alert saved → outputs/alerts/a4d51f79-b37e-448e-bea2-4bc736db4b63.json
{
  "alert_id": "a4d51f79-b37e-448e-bea2-4bc736db4b63",
  "schema_version": "1.0",
  "timestamp_utc": "2026-04-25T18:44:59.809604+00:00",
  "source": {
    "tile_path": "data/raw/10___1_S2A_20240317T085711_R1000_nonorm_bands20m.tiff",
    "sensor": "Sentinel-2",
    "bands_used": [
      "B11 (SWIR_1)",
      "B12 (SWIR_2)"
    ],
    "crs": "EPSG:32634",
    "bounds": {
      "left": 839520.0,
      "bottom": 2840020.0,
      "right": 849760.0,
      "top": 2850260.0
    }
  },
  "detection": {
    "plume_detected": true,
    "confidence_threshold": 0.5,
    "plume_pixel_count": 1426,
    "plume_coverage_pct": 0.544,
    "mean_plume_prob": 0.9652,
    "centroid_pixel": [
      241,
      272
    ],
    "centroid_geographic": [
      844340.0,
      2844820.0
    ]
  },
  "severity": {
    "class_id": 2,
    "label": "HIGH",
    "c